# Loading the data after processing in PyStructure

In [1]:
import matplotlib.pyplot as plt
import numpy as np
struct = np.load('Output_new/ngc5194_data_struct_3as_2025_11_15.npy',allow_pickle = True).item()
struct

{'source': 'ngc5194',
 'ra_deg': array([202.46277516, 202.46121737, 202.46184047, ..., 202.47087666,
        202.4715004 , 202.47212413]),
 'dec_deg': array([47.16802471, 47.16839157, 47.16839157, ..., 47.22231969,
        47.22231969, 47.22231969]),
 'dist_mpc': np.float64(8.0),
 'posang_deg': np.float64(172.0),
 'incl_deg': np.float64(20.0),
 'beam_as': 3.05,
 'rgal_as': array([99.7663018 , 99.34613694, 98.97492903, ..., 97.96561364,
        98.0604843 , 98.18205832]),
 'rgal_kpc': array([3.86944544, 3.85314931, 3.83875197, ..., 3.79960558, 3.80328515,
        3.80800041]),
 'rgal_r25': array([0.42854941, 0.42674457, 0.42515004, ..., 0.42081449, 0.42122201,
        0.42174424]),
 'theta_rad': array([-0.32803714, -0.37041402, -0.35450914, ...,  2.96002062,
         2.94350136,  2.92701852]),
 'INT_VAL_SIGMA_SFR24UM+HA': array([nan, nan, nan, ..., nan, nan, nan]),
 'INT_UC_SIGMA_SFR24UM+HA': array([nan, nan, nan, ..., nan, nan, nan]),
 'INT_UNIT_SIGMA_SFR24UM+HA': 'Msun/yr/kpc/kpc',
 '

## Calculation of beam area for the hexagonal grid a refeence image for the calculation of the hexagonal grid can be found on the PyStructure github page

In [2]:
#distance array
rgal_kpc = struct["rgal_kpc"]
#combined sfr from 24um + Ha from previous calculation
uc_sfr = struct["INT_UC_SIGMA_SFR24UM+HA"]
#sfr from JWST Paα
flux_PAA = struct["INT_VAL_F_PAA"]

theta_rad = struct["incl_deg"]

#calculating the pixel area
import numpy as np
import astropy.units as u

# inputs
theta_arcsec = 3.5            # beam FWHM in arcsec
total_pixels = 9015
ra_span_arcsec = 0.0093 * 3600  # deg -> arcsec 
dec_span_arcsec = 0.05429 * 3600 # deg -> arcsec

# compute areas
beam_area_arcsec2 = np.pi * theta_arcsec**2 / (4 * np.log(2))
image_area_arcsec2 = ra_span_arcsec * dec_span_arcsec
pix_area_arcsec2 = image_area_arcsec2 / total_pixels
pixels_per_beam = beam_area_arcsec2 / pix_area_arcsec2

# convert pixel angular area -> physical area at distance D
D_Mpc = 8.58
D_cm = (D_Mpc * 1e6 * u.pc).to(u.cm).value
# 1 arcsec = 1/206265 rad, so linear scale = D * (arcsec_in_rad)
area_pix_cm2 = pix_area_arcsec2 * (D_cm / 206265.0)**2

# assign astropy Quantity (this matches area_pix.value used in your code)
area_pix = area_pix_cm2 * u.cm**2

print("beam_area (arcsec^2):", beam_area_arcsec2)
print("pixel_area (arcsec^2):", pix_area_arcsec2)
print("pixels per beam:", pixels_per_beam)
print("pixel_area (cm^2):", area_pix_cm2)
print("pixel_area (pc^2):", (area_pix.to(u.pc**2)).value)



beam_area (arcsec^2): 13.88035293434578
pixel_area (arcsec^2): 0.7258419434276205
pixels per beam: 19.123106703918246
pixel_area (cm^2): 1.1958222905827292e+40
pixel_area (pc^2): 1255.9303461975367


## Calculation of SFR and $\Sigma_{SFR}$ for all the tracers 

### For JWST $P_{\alpha}$

In [3]:
import numpy as np

# -------------------------------
# Constants
# -------------------------------
d = 8.58  # Mpc
d_err = 0.1
lambda_pivot = 1.874  # um
delta_lambda = 0.024  # um
mpc_to_cm = 3.086e24  # cm
kpc_to_cm = 3.086e21  # cm
jy_to_ergs = 1e-23  # erg s^-1 cm^-2 Hz^-1
calzetti_sfr_pa_alpha = 4.26e-41  # M☉ yr^-1 / (erg s^-1)
c = 2.99792458e8  # m/s

# ---------------------------------
# Pixel physical area (hexagon)
# ---------------------------------
area_pix = 1.1958222905827292e40  # cm²  (hexagonal effective area)
area_pix_kpc2 = area_pix / (kpc_to_cm**2)  # convert to kpc²

# ---------------------------------
# Pixel angular sampling (square grid)
# ---------------------------------
# Side length per pixel in radians

deg_to_rad = np.pi / 180.0

pix_size_rad = (1.11e-5 * deg_to_rad)  # (1.11×10^-5 in both x and y)
D_cm = d * mpc_to_cm  # distance in cm

# area on the sky for a square pixel
area_square_pix_cm2 = (D_cm * pix_size_rad)**2
area_square_pix_kpc2 = area_square_pix_cm2 / (kpc_to_cm**2)

# --------------------------------------------
# Define calculation functions
# --------------------------------------------
def lum_cal_from_flux(Fv):
    """Return luminosity [erg s⁻¹] given flux density [Jy]"""
    Fl = Fv * jy_to_ergs * (c * 1e6 / lambda_pivot**2)
    F = Fl * delta_lambda / 0.903
    L = 4 * np.pi * (d * mpc_to_cm)**2 * F
    return L

def sfr_cal_from_flux(Fv):
    """Return total SFR [M☉ yr⁻¹] given flux density [Jy]"""
    L = lum_cal_from_flux(Fv)
    SFR = calzetti_sfr_pa_alpha * L
    return SFR

def sigma_sfr_cal_from_flux(Fv, area_pix_cm2):
    """Return SFR surface density [M☉ yr⁻¹ cm⁻²]"""
    SFR = sfr_cal_from_flux(Fv)
    return SFR / area_pix_cm2

# --------------------------------------------
# Apply to your JWST data
# --------------------------------------------
# struct["INT_VAL_F_PAA"] in Jy kpc^-2
int_jwst_f = np.array(struct["INT_VAL_F_PAA"])

# Step 1: Convert intensity (Jy/kpc²) flux per pixel (Jy)
# using the SQUARE pixel area, not the hexagon
flux_per_pix_Jy = int_jwst_f * area_pix_kpc2  # using area_pix_kpc2 and removing area_square_pix_kpc2

# Step 2: Compute luminosity, SFR, and ΣSFR
L_array = lum_cal_from_flux(flux_per_pix_Jy)                      # erg/s
SFR_array = sfr_cal_from_flux(flux_per_pix_Jy)                    # M☉/yr
sig_SFR_array = sigma_sfr_cal_from_flux(flux_per_pix_Jy, area_pix)  # M☉/yr/cm²

# Step 3: Convert ΣSFR from cm⁻² → kpc⁻²
sig_SFR_kpc2 = sig_SFR_array * (kpc_to_cm**2)  # M☉/yr/kpc²

# --------------------------------------------
# Combine results
# --------------------------------------------
results = np.column_stack((L_array, SFR_array, sig_SFR_array, sig_SFR_kpc2))

results_dict = {
    "Luminosity_erg_s": L_array,
    "SFR_Msun_yr": SFR_array,
    "Sigma_SFR_Msun_yr_cm2": sig_SFR_array,
    "Sigma_SFR_Msun_yr_kpc2": sig_SFR_kpc2,
    "Flux_Jy_per_pix": flux_per_pix_Jy,
    "Pixel_area_cm2_hex": area_pix,
    "Square_pix_area_kpc2": area_square_pix_kpc2,
}

# --------------------------------------------
# Print summary
# --------------------------------------------
print(f"Distance: {d:.2f} Mpc")
print(f"Square pixel area (kpc²): {area_square_pix_kpc2:.3e}")
print(f"Hexagon pixel area (cm²): {area_pix:.3e}")
print(f"Hexagon pixel area (kpc²): {area_pix_kpc2:.3e}\n")

print("Results (first 5 pixels):")
for key, val in results_dict.items():
    if isinstance(val, np.ndarray):
        print(f"{key}: {val[:5]}")


Distance: 8.58 Mpc
Square pixel area (kpc²): 2.763e-06
Hexagon pixel area (cm²): 1.196e+40
Hexagon pixel area (kpc²): 1.256e-03

Results (first 5 pixels):
Luminosity_erg_s: [7.85327593e+35 1.07899292e+36 8.09948383e+35 7.87568309e+35
 8.06644471e+35]
SFR_Msun_yr: [3.34549555e-05 4.59650982e-05 3.45038011e-05 3.35504100e-05
 3.43630545e-05]
Sigma_SFR_Msun_yr_cm2: [2.79765277e-45 3.84380677e-45 2.88536193e-45 2.80563510e-45
 2.87359207e-45]
Sigma_SFR_Msun_yr_kpc2: [0.02664316 0.03660609 0.02747844 0.02671917 0.02736636]
Flux_Jy_per_pix: [3.92888713e-06 5.39805480e-06 4.05206160e-06 3.94009713e-06
 4.03553257e-06]


### For 24 $\mu m$ and $H_{\alpha}$

In [4]:
import numpy as np

# Constants
c = 3e8  # m/s
lambda_24 = 24e-6  # m
nu_24 = c / lambda_24  # Hz

def calc_sigma_sfr(I_Halpha, I_24_MJy):
    """
    Calculate Sigma_SFR [M_sun yr^-1 kpc^-2] using Halpha + 24um intensities.
    
    struct: dictionary-like object with keys:
        - 'INT_VAL_HA_SINGS' [erg s^-1 cm^-2 sr^-1]
        - 'INT_VAL_24UM_HIRES' [MJy sr^-1]
    """

    # Convert I_24um from [MJy/sr] to [erg s^-1 cm^-2 sr^-1]
    I_24 = I_24_MJy * 1e6 * 1e-23 * nu_24  # erg s^-1 cm^-2 sr^-1

    # Compute Sigma_SFR [M_sun yr^-1 kpc^-2]
    sigma_SFR = (634 * I_Halpha + 0.00325 * I_24) * np.cos(theta_rad)

    return sigma_SFR



I_Halpha = struct["INT_VAL_HA_SINGS"]              # erg s^-1 cm^-2 sr^-1
I_24_MJy = struct["INT_VAL_24UM_HIRES"]            # MJy sr^-1

sigma_SFR = calc_sigma_sfr(I_Halpha, I_24_MJy)      


In [5]:
import numpy as np

# ----------------------------
# Constants
# ----------------------------
kpc_to_cm = 3.086e21  # cm in 1 kpc
area_pix = 1.1958222905827292e+40  # cm^2

# Convert pixel area from cm² to kpc²
area_pix_kpc2 = area_pix / (kpc_to_cm**2)

# ----------------------------
# Compute SFR [M☉ yr⁻¹] from ΣSFR [M☉ yr⁻¹ kpc⁻²]
# ----------------------------
SFR_from_sigma = sigma_SFR * area_pix_kpc2


In [6]:
import numpy as np

# Conversion constants
kpc_to_cm = 3.086e21  # cm per kpc

def sfr_from_sigma(sigma_sfr_kpc2, area_pix_cm2):
    """
    Convert ΣSFR [M☉ yr⁻¹ kpc⁻²] to SFR [M☉ yr⁻¹],
    given a pixel area in cm².
    """
    # Convert from kpc⁻² → cm⁻²
    sigma_sfr_cm2 = sigma_sfr_kpc2 / (kpc_to_cm**2)

    # Compute total SFR in one pixel
    sfr = sigma_sfr_cm2 * area_pix_cm2

    return sfr



sigma_sfr_input = struct["INT_VAL_SIGMA_SFR24UM+HA"]
area_pix = 1.1958222905827292e+40  # cm²


sfr_value = sfr_from_sigma(sigma_sfr_input, area_pix)


### For VLA 33GHz data

In [7]:
import numpy as np

def sfr_from_vla33(int_vla, Te=6300.0, nu=33.0, D_mpc=8.58, pix_scale_deg=5.55e-5):


    # --- Constants ---
    pc_to_cm = 3.0857e18
    D_cm = D_mpc * 1e6 * pc_to_cm

    # ---------------------------------
# Pixel physical area (hexagon)
# ---------------------------------
    area_pix = 1.1958222905827292e40  # cm²  (hexagonal effective area)
    area_pix_kpc2 = area_pix / (kpc_to_cm**2)  # convert to kpc²

    # --- Pixel area in kpc^2 ---
    pix_size_kpc = np.deg2rad(pix_scale_deg) * D_mpc * 1e3  # kpc per pixel
    pix_area_kpc2 = pix_size_kpc**2

    # --- Convert Jy/kpc^2 -> Jy ---
    S_nu_Jy = int_vla * area_pix_kpc2

    # --- Convert Jy to erg/s/cm^2/Hz ---
    S_nu_erg = S_nu_Jy * 1e-23

    # --- Luminosity: Lν = 4πD²Sν ---
    L_nu = 4 * np.pi * (D_cm**2) * S_nu_erg

    # --- Murphy et al. (2011) equation ---
    sfr = 4.6e-28 * (Te / 1e4)**(-0.45) * (nu)**0.1 * L_nu

    return sfr



# Compute SFR array
sfr_vla = sfr_from_vla33(struct["INT_VAL_VLA33GHZ"])

# Print a few results
print("Thermal SFR (M_sun/yr):")
print(sfr_vla)


Thermal SFR (M_sun/yr):
[-1.95013459e-05 -1.34912797e-05 -2.16472979e-05 ...  1.99602358e-05
  2.60814476e-05  6.71604290e-06]


In [8]:
import numpy as np

def sigma_sfr_from_vla33(int_vla, area_pix, Te=6300.0, nu=33.0, D_mpc=8.58):
    """
    Compute ΣSFR [M_sun yr^-1 kpc^-2] from VLA 33 GHz intensities (Jy/kpc^2)
    using Murphy et al. (2011).

    Parameters
    ----------
    int_vla : array_like
        Thermal intensities in Jy/kpc^2.
    area_pix : float
        Pixel area in cm^2.
    Te : float, optional
        Electron temperature in K (default=6300 K).
    nu : float, optional
        Frequency in GHz (default=33 GHz).
    D_mpc : float, optional
        Distance to the galaxy in Mpc (default=8.58 Mpc).

    Returns
    -------
    sigma_sfr : ndarray
        Star formation rate surface density in M_sun yr^-1 kpc^-2.
    """

    # --- Constants ---
    pc_to_cm = 3.0857e18
    D_cm = D_mpc * 1e6 * pc_to_cm
    kpc_to_cm = 3.0857e21



    # ---------------------------------
# Pixel physical area (hexagon)
# ---------------------------------
    area_pix = 1.1958222905827292e40  # cm²  (hexagonal effective area)
    area_pix_kpc2 = area_pix / (kpc_to_cm**2)  # convert to kpc²

    
    # --- Convert pixel area from cm² → kpc² ---
    area_pix_kpc2 = area_pix / (kpc_to_cm**2)

    # --- Convert Jy/kpc² → Jy ---
    S_nu_Jy = int_vla * area_pix_kpc2

    # --- Jy → erg s^-1 cm^-2 Hz^-1 ---
    S_nu_erg = S_nu_Jy * 1e-23

    # --- Luminosity Lν [erg s^-1 Hz^-1] ---
    L_nu = 4 * np.pi * (D_cm**2) * S_nu_erg

    # --- Murphy et al. (2011) total SFR [M_sun/yr] ---
    SFR = 4.6e-28 * (Te / 1e4)**(-0.45) * (nu)**0.1 * L_nu

    # --- Surface density ΣSFR = SFR / area_pix_kpc² ---
    sigma_SFR = SFR / area_pix_kpc2

    return sigma_SFR


# ----------------------------
# Example usage
# ----------------------------
area_pix = 1.1958222905827292e+40  # cm² 


sigma_sfr_vla = sigma_sfr_from_vla33(struct["INT_VAL_VLA33GHZ"], area_pix)

print("ΣSFR [M_sun yr^-1 kpc^-2]:")
print(sigma_sfr_vla)


ΣSFR [M_sun yr^-1 kpc^-2]:
[-0.01553066 -0.01074431 -0.01723967 ...  0.01589611  0.02077098
  0.00534858]


### For VENGA $H_{\alpha}$ data

In [14]:
import numpy as np

def calculate_sfr_extinction_corrected(
    flux_map_Ha_per_kpc2,
    flux_map_Hb_per_kpc2,
    distance_Mpc,
    area_hex_cm2
):
    """
    Compute extinction-corrected SFR and Σ_SFR for VENGA Hα data.
    """

    # Constants
    kpc_to_cm = 3.086e21
    Mpc_to_cm = 3.086e24
    distance_cm = distance_Mpc * Mpc_to_cm

    # Reddening curve values (O'Donnell 1994)
    k_Ha = 2.52
    k_Hb = 3.66

    # Intrinsic Case B recombination Hα/Hβ ratio
    R_intrinsic = 2.86

    # Conversion factor log10 C_Hα = −41.26 → SFR = C_Ha * L_Hα
    C_Ha = 10.0**(-41.26)

    # Convert hexagon area to kpc^2
    area_hex_kpc2 = area_hex_cm2 / (kpc_to_cm**2)

    # Convert flux per kpc^2 → flux per hexagon in erg/s/cm^2
    F_Ha = flux_map_Ha_per_kpc2 * area_hex_kpc2
    F_Hb = flux_map_Hb_per_kpc2 * area_hex_kpc2

    # Convert flux → luminosity
    L_Ha = 4 * np.pi * distance_cm**2 * F_Ha
    L_Hb = 4 * np.pi * distance_cm**2 * F_Hb

    # Avoid division warnings
    ratio_obs = np.where(L_Hb > 0, L_Ha / L_Hb, np.nan)

    # Compute E(B-V) using Eq 6.2
    EBV = (2.5 / (k_Hb - k_Ha)) * np.log10(ratio_obs / R_intrinsic)

    # Clip negative reddening (common practice)
    EBV = np.where(np.isfinite(EBV), np.maximum(EBV, 0.0), 0.0)

    # Extinction correction (Eq 6.3)
    L_Ha_corr = L_Ha * 10**(0.4 * k_Ha * EBV)

    # Compute SFR and Sigma_SFR
    sfr = C_Ha * L_Ha_corr
    sigma_sfr = sfr / area_hex_kpc2

    return sfr, sigma_sfr


In [18]:
# Load your VENGA maps (already in erg/s/cm²/kpc²)
flux_map_Ha = struct["INT_VAL_VENGA_HA"]
flux_map_Hb = struct["INT_VAL_VENGA_HB"]

distance_M51 = 8.6  # Mpc
area_hex_cm2 = 1.1958222905827292e40  # cm²

# Compute extinction-corrected SFR and Σ_SFR
sfr_venga_extinction_corr, sigma_sfr_venga_extinction_corr = \
    calculate_sfr_extinction_corrected(
        flux_map_Ha,
        flux_map_Hb,
        distance_M51,
        area_hex_cm2
    )
